# PV-Hochlauf Reichshof — Walkthrough

Dieses Notebook zeigt für **Reichshof** (AGS 05374040), **wo** PV auf den Dächern Platz hat
und **wie viel** PV-Leistung bis 2050 dazukommt.

- **Karte:** das Dach-PV-Potenzial je Gebäude — die räumliche Verteilung über die Gemeinde.
- **Band:** der zeitliche Hochlauf der installierten Leistung (MWp) als Unsicherheitsband.

**Methode (kurz):** Ein probabilistisches Diffusions-Ensemble fittet 12 Wachstumskurven an
vergleichbare Gemeinden, kombiniert sie zu einem Band und gewichtet sie danach, wie gut sie
die Vergangenheit getroffen hätten.
Quellen: Zielonka et al. 2023 ([doi:10.1093/pnasnexus/pgad321](https://doi.org/10.1093/pnasnexus/pgad321)),
WIS: Bracher et al. 2021 ([doi:10.1371/journal.pcbi.1008618](https://doi.org/10.1371/journal.pcbi.1008618)).
Daten: MaStR (Bestand), InfDB/LoD2 (Gebäude, Dachflächen).

> Das Notebook lädt vorberechnete Referenzdaten aus `data/` und läuft offline in Sekunden.
> Neu rechnen: siehe letzter Abschnitt.

## 1 · Setup

In [ ]:
import json
from math import cos, radians
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

DATA = Path('data')
TEAL, AMBER, INK = '#0e7c73', '#d98a1f', '#16202a'

history = pd.read_csv(DATA / 'reichshof_history.csv')          # beobachteter Bestand (MaStR)
band    = pd.read_csv(DATA / 'reichshof_band.csv')             # Ensemble-Band (MWp)
meta    = json.loads((DATA / 'reichshof_meta.json').read_text())
geo     = pd.read_csv(DATA / 'reichshof_buildings_geo.csv')    # Wohngebäude + Dach-PV-Potenzial
boundary= json.loads((DATA / 'reichshof_boundary.geojson').read_text())

anker = history['mwp_kum'].iloc[-1]
print(f"{len(geo):,} Wohngebäude | Σ Dach-PV-Potenzial {geo['kwp_potenzial'].sum()/1000:.0f} MWp")
print(f"Bestand heute: {anker:.1f} MWp aus {int(history['anlagen_kum'].iloc[-1]):,} Anlagen")

## 2 · Karte: PV-Verteilung über Reichshof

Jeder Punkt ist ein Wohngebäude, eingefärbt nach seinem **Dach-PV-Potenzial** (kWp, geschätzt aus
der LoD2-Dachfläche). So sieht man, wo in der Gemeinde die PV-Kapazität sitzt: dicht in den
Ortslagen, dünn im ländlichen Umland.

> Die **Modell**-Zuteilung einzelner Anlagen ist bewusst räumlich neutral (keine Geo-Annahmen).
> Darum zeigt die Karte das reale **Potenzial** je Gebäude, nicht eine erfundene Punktverteilung.

In [ ]:
def rings(g):
    if g['type'] == 'Polygon':      return [g['coordinates']]
    if g['type'] == 'MultiPolygon': return [p for p in g['coordinates']]
    return []

fig, ax = plt.subplots(figsize=(8.2, 8))
# Gemeindegrenze
for poly in rings(boundary):
    for ring in poly:
        xy = np.array(ring)
        ax.plot(xy[:,0], xy[:,1], color=INK, lw=1.1, alpha=.55)
# Gebäude nach Potenzial (oben gekappt, damit die EFH-Variation sichtbar bleibt)
vmax = float(np.nanpercentile(geo['kwp_potenzial'], 95))
sc = ax.scatter(geo['lon'], geo['lat'], c=geo['kwp_potenzial'], cmap='YlGnBu',
                s=9, alpha=.75, norm=Normalize(0, vmax), linewidths=0)
cb = fig.colorbar(sc, ax=ax, shrink=.6, pad=.02)
cb.set_label('Dach-PV-Potenzial je Gebäude [kWp]')
ax.set_aspect(1 / cos(radians(51)))  # geographisch korrektes Seitenverhältnis
ax.set_xlabel('Länge [°O]'); ax.set_ylabel('Breite [°N]')
ax.set_title('Reichshof — Dach-PV-Potenzial je Wohngebäude')
ax.margins(.02); plt.tight_layout(); plt.show()

### Wo konzentriert sich die Kapazität?

Dieselben Gebäude, aber als Dichte: aufsummiertes Potenzial je Raster­zelle (MWp). Die Spitzen
markieren die Siedlungskerne — dort entsteht der Großteil des Hochlaufs.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 8))
hb = ax.hexbin(geo['lon'], geo['lat'], C=geo['kwp_potenzial']/1000, reduce_C_function=np.sum,
               gridsize=38, cmap='YlGnBu', mincnt=1, linewidths=.2)
for poly in rings(boundary):
    for ring in poly:
        xy = np.array(ring); ax.plot(xy[:,0], xy[:,1], color=INK, lw=1.1, alpha=.55)
cb = fig.colorbar(hb, ax=ax, shrink=.6, pad=.02); cb.set_label('Σ Dach-PV-Potenzial je Zelle [MWp]')
ax.set_aspect(1 / cos(radians(51)))
ax.set_xlabel('Länge [°O]'); ax.set_ylabel('Breite [°N]')
ax.set_title('Reichshof — PV-Potenzialdichte')
ax.margins(.02); plt.tight_layout(); plt.show()

## 3 · Der beobachtete PV-Bestand

Reichshof hat heute rund **9,7 MWp** aus gut 1.300 Anlagen. Der Zubau hat sich ab ~2021
beschleunigt — der typische S-Kurven-Anlauf.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.1))
ax.plot(history['jahr'], history['mwp_kum'], color=TEAL, lw=2, marker='o', ms=3, label='beobachtet (MaStR)')
ax.scatter([history['jahr'].iloc[-1]], [anker], color=AMBER, zorder=5, s=55,
           label=f'Anker {int(history["jahr"].iloc[-1])}: {anker:.1f} MWp')
ax.set_xlabel('Jahr'); ax.set_ylabel('kumulierte PV-Leistung [MWp]')
ax.set_title('PV-Bestand Reichshof'); ax.grid(alpha=.3); ax.legend(); ax.set_xlim(2005, 2025)
plt.tight_layout(); plt.show()

## 4 · Das Hochlaufband bis 2050

Aus dem Anker heraus projiziert das Ensemble die Entwicklung — mittlerer Pfad (p50) und
Unsicherheitsbereich (p05–p95). Zum Vergleich der beobachtete Verlauf.

In [ ]:
fut = band[band['jahr'] >= 2015]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.fill_between(fut['jahr'], fut['p05_MWp'], fut['p95_MWp'], color=TEAL, alpha=.15, label='p05–p95')
ax.fill_between(fut['jahr'], fut['p25_MWp'], fut['p75_MWp'], color=TEAL, alpha=.28, label='p25–p75')
ax.plot(fut['jahr'], fut['p50_MWp'], color=TEAL, lw=2.2, label='p50 (Median)')
ax.plot(history['jahr'], history['mwp_kum'], color=INK, lw=1.6, ls='--', marker='o', ms=3, label='beobachtet')
p50_2050 = band.loc[band['jahr']==2050, 'p50_MWp'].iloc[0]
ax.annotate(f'{p50_2050:.1f} MWp', xy=(2050, p50_2050), xytext=(2044, p50_2050+3),
            color=TEAL, fontsize=10, fontweight='bold')
ax.set_xlabel('Jahr'); ax.set_ylabel('PV-Leistung [MWp]')
ax.set_title('PV-Hochlauf Reichshof — Ensemble-Band')
ax.grid(alpha=.3); ax.legend(loc='upper left'); ax.set_xlim(2015, 2050)
plt.tight_layout(); plt.show()

### Stützjahre

In [ ]:
stuetz = [2025, 2030, 2035, 2040, 2045, 2050]
tab = band[band['jahr'].isin(stuetz)].set_index('jahr')[['p05_MWp','p25_MWp','p50_MWp','p75_MWp','p95_MWp']].round(1)
tab.columns = ['p05','p25','p50','p75','p95']; tab

## 5 · Welche Modelle tragen das Band?

Für Reichshof dominieren die **Bi-S-Kurven** (zwei überlagerte Wachstumsschübe) — passend zum
beschleunigten Zubau der letzten Jahre.

In [ ]:
w = pd.Series(meta['model_weights']).sort_values()
colors = [AMBER if n.startswith('Bi-') else TEAL for n in w.index]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.barh(w.index, w.values*100, color=colors)
ax.set_xlabel('Gewicht [%]'); ax.set_title('Modellgewichte aus dem Hindcast')
bi = sum(v for n,v in meta['model_weights'].items() if n.startswith('Bi-'))
ax.text(0.98, 0.04, f'Bi-S-Kurven zusammen: {bi*100:.0f} %', transform=ax.transAxes,
        ha='right', color=AMBER, fontweight='bold')
ax.grid(alpha=.3, axis='x'); plt.tight_layout(); plt.show()

## 6 · Einordnung

- **Karte × Band:** Die Karte zeigt, *wo* Kapazität sitzt (Σ 46 MWp Dachpotenzial der Wohngebäude
  aus der InfDB), das Band, *wie viel* davon über die Zeit genutzt wird — heute ~9,7 MWp,
  2050 im Median ~22,5 MWp.
- Karte und Band nutzen **dasselbe** Dachpotenzial (InfDB, wie AP 2 und pylovo). Der Hochlauf bleibt
  klar darunter; die Obergrenze bindet nicht (mit dem früheren BUILDA-Wert 68,8 MWp ist das Band gleich).
- **Methodik:** `docs/methodik_pv.md`, `docs/methodik_umstellung_zielonka.md`.
  AP-2-Kopplung (Objekttypen): `examples/`.

## 7 · Neu rechnen (optional)

Die Referenzdaten stammen aus einem vollständigen Lauf. Neu rechnen (**~19 min**):

```bash
python pv_hochlauf/run_phase1.py --target 05374040                        # Dachpotenzial aus der InfDB
python pv_hochlauf/run_phase1.py --target 05374040 --saturation builda    # Vergleich: BUILDA
```

Braucht OEDS (MaStR) und die lokale InfDB mit der Gemeinde (Port 54328).

Ergebnis: `band_05374040.csv` + `meta_05374040.json`; die Gebäude-Geodaten entstehen aus der
InfDB (`basedata.buildings` + `building_surface_area`).